# exp034_public_sel15_pf_meta_stack train

Regenerate an exp026-style pseudo-tail anchor on exp029 pseudo-hidden PF rows, then audit fixed blends and shallow residual meta models.

## Contents

1. Setup and configuration
2. Input artifact check
3. Meta-stack audit
4. Metrics and artifacts


## 1. Setup and configuration

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

from meta_stack_audit import resolve_feature_path, run_audit
from settings import EXPERIMENT_NAME, ExperimentPaths, get_nested, load_config

paths = ExperimentPaths()
paths.ensure_output_dirs()
config = load_config()
feature_path = resolve_feature_path(paths, Path(get_nested(config, "data.feature_path")))

print("Experiment:", EXPERIMENT_NAME)
print("Route:", get_nested(config, "experiment.route"))
print("Parent:", get_nested(config, "lineage.parent"))
print("Feature path:", feature_path)
print("Target:", get_nested(config, "model.target"))
print("Meta features:", len(get_nested(config, "model.features") or []))
print("Pseudo-tail variant:", get_nested(config, "audit.training_variants.selected_variant"))


## 2. Input artifact check

In [ ]:
if not feature_path.exists():
    raise FileNotFoundError(f"exp029 feature artifact not found: {feature_path}")

preview = pd.read_csv(feature_path, nrows=5)
required_preview_cols = [
    "well_id",
    "fold",
    "cutoff_row",
    "row_idx",
    "eval_step",
    "target_tvt",
    "last_anchor_tvt",
    "pf_pred",
    "beam_pred",
]
print("Preview rows:", len(preview))
print("Columns:", len(preview.columns))
display(preview[required_preview_cols])


## 3. Meta-stack audit

In [ ]:
summary = run_audit(paths, config, feature_path)
print(json.dumps({
    "rows": summary["rows"],
    "wells": summary["wells"],
    "required_control_by_audit": summary["required_control_by_audit"],
    "best_original_fold_candidate": summary["best_original_fold_candidate"],
    "best_original_fold_cv": summary["best_original_fold_cv"],
    "best_well_hash_candidate": summary["best_well_hash_candidate"],
    "best_well_hash_cv": summary["best_well_hash_cv"],
    "selected_candidate": summary["selected_candidate"],
    "meta_stack_supported": summary["meta_stack_supported"],
}, indent=2))


## 4. Metrics and artifacts

In [ ]:
metrics = pd.read_csv(paths.artifacts_dir / "meta_stack_metrics.csv")
buckets = pd.read_csv(paths.artifacts_dir / "meta_stack_bucket_metrics.csv")
splits = pd.read_csv(paths.artifacts_dir / "meta_stack_split_metrics.csv")
display(metrics.head(30))
display(buckets.head(20))
display(splits.head(20))
print("Metrics written:", paths.metrics_path)
print("Artifacts written:", paths.artifacts_dir)
